In [ ]:
# === 1. Import thư viện ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

import os
from docx import Document
import re

import zipfile
import json
from lxml import etree

d:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# finetune

In [ ]:
# === 2. Đọc dữ liệu CSV đã gắn nhãn ===
df = pd.read_csv("src_data/all_labeled_questions.csv")  # 🔹 đường dẫn tới file bạn tải lên

# Kiểm tra dữ liệu
print(df.head())

# === 3. Ánh xạ nhãn thành số ===
label_map = {"question": 0, "answer": 1}
df["label_id"] = df["label"].map(label_map)

# === 4. Chia tập train/test ===
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

# === 5. Chuẩn bị tokenizer ===
model_name = "vinai/phobert-base"  # ⚠️ Nếu dữ liệu là tiếng Việt, nên đổi thành "bert-base-multilingual-cased" hoặc "PhoBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['content'], truncation=True, padding='max_length', max_length=128)

# === 6. Tạo dataset cho HuggingFace ===
train_dataset = Dataset.from_pandas(train_df[['content', 'label_id']])
test_dataset = Dataset.from_pandas(test_df[['content', 'label_id']])

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# === 7. Load mô hình gốc ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_map))

# === 8. Cấu hình huấn luyện ===
training_args = TrainingArguments(
    output_dir="./model_output",
    do_eval=True,   # để mô hình vẫn chạy đánh giá trên eval_dataset
    save_total_limit=2,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # bật FP16 nếu có GPU
    report_to="none"
)

# === 9. Huấn luyện ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# === 10. Lưu mô hình đã fine-tune ===
save_path = "/content/bert_question_answer_classifier"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Fine-tune hoàn tất. Mô hình đã lưu tại:", save_path)
print("Label map:", label_map)


In [1]:
#!/usr/bin/env python3
"""
Selective extractor (strict, full): trích xuất/chuyển đổi chỉ những ảnh được chọn với
dedupe và ánh xạ canonical chính xác để mọi rId trong JSON đầu ra trỏ tới file tồn tại.

Cách dùng:
    python extract_docx_selective_strict.py path/to/file.docx

- JSON đầu ra cho mỗi paragraph chứa:
    - math: danh sách OMML string
    - images: danh sách nhóm ảnh với rIds, img_rels, sha1, saved_path, rId_to_saved, reasons
"""

import zipfile
import os
import json
import posixpath
import shutil
import subprocess
import hashlib
import re
from lxml import etree

# Namespace XML
NAMESPACES = {
    "m": "http://schemas.openxmlformats.org/officeDocument/2006/math",
    "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "v": "urn:schemas-microsoft-com:vml",
    "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
    "pic": "http://schemas.openxmlformats.org/drawingml/2006/picture",
    "wp": "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing",
    "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
}

def _r_attr(name):
    return "{%s}%s" % (NAMESPACES["r"], name)

def _resolve_target_posix(target):
    # target in rels is often like "media/image1.wmf" -> join with "word"
    return posixpath.normpath(posixpath.join("word", target))

def _find_imagemagick_cmd():
    """
    Try to locate ImageMagick CLI. Prefer 'magick' (modern IM). If only 'convert' found,
    ensure it's from ImageMagick (on Windows, skip C:\Windows\System32\convert.exe).
    """
    for cmd in ("magick", "convert"):
        path = shutil.which(cmd)
        if not path:
            continue
        # On Windows 'convert.exe' in System32 is not ImageMagick
        if os.name == "nt" and os.path.basename(path).lower() == "convert.exe":
            parent = os.path.dirname(path).lower()
            # ensure path looks like ImageMagick install
            if "imagemagick" in path.lower() or "imagemagick" in parent:
                return cmd
            else:
                # skip the System convert.exe
                continue
        return cmd
    return None

def convert_to_png(src_path, out_dir, imagemagick_cmd=None, debug=False):
    """
    Convert ANY format (PNG/JPG/BMP/TIF/WMF/EMF/SVG...) to PNG via ImageMagick.
    Returns output PNG path on success, or None on failure.
    """
    ext = os.path.splitext(src_path)[1].lower()

    # If already PNG, return original
    if ext == ".png":
        if debug:
            print(f"[convert] already png: {src_path}")
        return src_path

    # Find ImageMagick
    if not imagemagick_cmd:
        imagemagick_cmd = _find_imagemagick_cmd()

    if not imagemagick_cmd:
        if debug:
            print("[convert] ImageMagick không tìm thấy; bỏ qua convert cho", src_path)
        return None

    # Ensure output directory exists
    os.makedirs(out_dir, exist_ok=True)

    base = os.path.splitext(os.path.basename(src_path))[0]
    out_name = f"{base}.png"
    png_path = os.path.join(out_dir, out_name)

    # avoid name collisions
    i = 1
    while os.path.exists(png_path):
        png_path = os.path.join(out_dir, f"{base}_{i}.png")
        i += 1

    # Build command
    # For modern ImageMagick: magick "in" "out"
    # For older: convert "in" "out"
    cmd = [imagemagick_cmd, src_path, png_path]

    if debug:
        print("[convert] Chạy lệnh:", " ".join(cmd))

    try:
        # run and capture stderr/stdout for debugging
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)

        if proc.returncode != 0:
            if debug:
                try:
                    stderr = proc.stderr.decode("utf-8", errors="ignore")
                except Exception:
                    stderr = str(proc.stderr)
                print("[convert] ImageMagick trả lỗi:", stderr)
            # remove possibly created file
            if os.path.exists(png_path):
                try:
                    os.remove(png_path)
                except Exception:
                    pass
            return None

        if os.path.exists(png_path):
            if debug:
                print("[convert] Thành công:", png_path)
            return png_path
        else:
            if debug:
                print("[convert] Không thấy file output sau khi chạy:", png_path)
            return None

    except FileNotFoundError:
        if debug:
            print("[convert] ImageMagick executable không tìm thấy (FileNotFoundError).")
        return None
    except Exception as e:
        if debug:
            print("[convert] Exception khi convert:", str(e))
        return None

def _sha1_bytes(b):
    h = hashlib.sha1()
    h.update(b)
    return h.hexdigest()

def _merge_images_by_hash(images_info, canonical_by_sha1, debug=False):
    groups = {}
    order = []
    for img in images_info:
        key = img.get("sha1") or img.get("img_rel") or img.get("rId")
        if key not in groups:
            groups[key] = {
                "rIds": [],
                "img_rels": [],
                "exts": set(),
                "sizes": [],
                "sha1": img.get("sha1"),
                "alt_texts": set(),
                "selected": False,
                "saved_path": None,
                "reasons": set(),
                "rId_to_saved": {}
            }
            order.append(key)
        g = groups[key]
        if img.get("rId") and img["rId"] not in g["rIds"]:
            g["rIds"].append(img["rId"])
        if img.get("img_rel") and img["img_rel"] not in g["img_rels"]:
            g["img_rels"].append(img["img_rel"])
        if img.get("ext"):
            g["exts"].add(img["ext"])
        if img.get("size"):
            g["sizes"].append(img["size"])
        if img.get("alt"):
            g["alt_texts"].add(img["alt"])
        if img.get("selected"):
            g["selected"] = True
        if img.get("saved_path") and not g["saved_path"]:
            g["saved_path"] = img["saved_path"]
        if img.get("rId"):
            g["rId_to_saved"][img["rId"]] = img.get("saved_path")
        for reason in img.get("reasons", []):
            g["reasons"].add(reason)

    merged = []
    for key in order:
        g = groups[key]
        canonical = g["saved_path"] or canonical_by_sha1.get(g["sha1"])
        if not canonical:
            for v in g["rId_to_saved"].values():
                if v:
                    canonical = v
                    break
        rId_to_saved_norm = {rid: canonical if canonical else g["rId_to_saved"].get(rid) for rid in g["rIds"]}
        merged.append({
            "rIds": g["rIds"],
            "img_rels": g["img_rels"],
            "exts": list(g["exts"]),
            "sizes": g["sizes"],
            "sha1": g["sha1"],
            "alts": list(g["alt_texts"]),
            "selected": g["selected"],
            "saved_path": canonical,
            "rId_to_saved": rId_to_saved_norm,
            "reasons": list(g["reasons"]),
        })
    if debug:
        print(f"[merge] Đã gom {len(images_info)} -> {len(merged)} nhóm ảnh")
    return merged

QUESTION_REGEX = re.compile(r'^\s*(Câu|Cau|Question)\b\s*\d+', flags=re.IGNORECASE)
KEYWORD_IN_ALT = ("hình", "hinh", "ảnh", "anh", "figure", "fig", "image")
PROXIMITY_WINDOW_DEFAULT = 3
MIN_SIZE_BYTES_DEFAULT = 5
LARGE_SIZE_BYTES_DEFAULT = 20_000
VERY_LARGE_FACTOR = 2

def extract_docx_selective_strict(docx_path, out_folder="extracted_strict",
                                  convert_to_png_flag=False,
                                  proximity_window=PROXIMITY_WINDOW_DEFAULT,
                                  min_size_bytes=MIN_SIZE_BYTES_DEFAULT,
                                  large_size_bytes=LARGE_SIZE_BYTES_DEFAULT,
                                  debug=False):
    os.makedirs(out_folder, exist_ok=True)
    media_out = os.path.join(out_folder, "media")
    os.makedirs(media_out, exist_ok=True)
    results = []

    with zipfile.ZipFile(docx_path) as zf:
        try:
            document_xml = etree.fromstring(zf.read("word/document.xml"))
        except KeyError:
            raise FileNotFoundError("word/document.xml không tìm thấy trong .docx")

        rels_map = {}
        try:
            rels_xml = etree.fromstring(zf.read("word/_rels/document.xml.rels"))
            for rel in rels_xml.findall("rel:Relationship", NAMESPACES):
                rid = rel.get("Id") or rel.get("ID")
                target = rel.get("Target")
                target_mode = rel.get("TargetMode")
                if rid and target:
                    rels_map[rid] = {"target": target, "target_mode": target_mode}
            if debug:
                print("[info] rels_map size:", len(rels_map))
        except KeyError:
            if debug:
                print("[info] Không tìm thấy word/_rels/document.xml.rels")
            rels_map = {}

        para_nodes = document_xml.findall(".//w:p", NAMESPACES)
        question_paras = set()
        para_texts = []
        for i, p in enumerate(para_nodes):
            texts = [t.text for t in p.findall(".//w:t", NAMESPACES) if t.text]
            txt = "".join(texts).strip()
            para_texts.append(txt)
            if QUESTION_REGEX.search(txt):
                question_paras.add(i)
                if debug:
                    print(f"[info] paragraph {i} nghi là câu hỏi: {txt[:80]}")

        extracted_map = {}
        canonical_by_sha1 = {}
        imagemagick_cmd = _find_imagemagick_cmd() if convert_to_png_flag else None
        if debug and convert_to_png_flag:
            print("[info] ImageMagick command:", imagemagick_cmd)

        def _get_image_alt_from_node(node):
            cur = node
            while cur is not None:
                docpr = cur.find(".//wp:docPr", NAMESPACES)
                if docpr is not None:
                    name = docpr.get("name") or ""
                    descr = docpr.get("descr") or ""
                    return (name + " " + descr).strip()
                cNvPr = cur.find(".//pic:cNvPr", NAMESPACES)
                if cNvPr is not None:
                    name = cNvPr.get("name") or ""
                    descr = cNvPr.get("descr") or ""
                    return (name + " " + descr).strip()
                cur = cur.getparent()
            return ""

        for pi, p in enumerate(para_nodes):
            text_parts = [t.text for t in p.findall(".//w:t", NAMESPACES) if t.text]
            text = "".join(text_parts).strip()

            omml_nodes = p.findall(".//m:oMath", NAMESPACES) + p.findall(".//m:oMathPara", NAMESPACES)
            math_list = [{"omml": etree.tostring(node, encoding="unicode")} for node in omml_nodes]

            images_info = []
            seen_rids = set()

            def _process_rid(rid, xml_node):
                if not rid or rid in seen_rids:
                    return
                seen_rids.add(rid)
                info = rels_map.get(rid)
                if not info:
                    images_info.append({
                        "rId": rid,
                        "img_rel": None,
                        "selected": False,
                        "reasons": ["no_rels"],
                        "sha1": None,
                        "saved_path": None,
                    })
                    return
                if info.get("target_mode") == "External":
                    images_info.append({
                        "rId": rid,
                        "img_rel": info.get("target"),
                        "selected": False,
                        "reasons": ["external"],
                        "sha1": None,
                        "saved_path": None,
                    })
                    return

                img_rel = _resolve_target_posix(info["target"])
                try:
                    raw = zf.read(img_rel)
                except KeyError:
                    images_info.append({
                        "rId": rid,
                        "img_rel": img_rel,
                        "selected": False,
                        "reasons": ["not_in_zip"],
                        "sha1": None,
                        "saved_path": None,
                    })
                    return

                size = len(raw)
                sha1 = _sha1_bytes(raw)
                alt_text = _get_image_alt_from_node(xml_node) or ""
                alt_text_l = alt_text.lower()
                ext = os.path.splitext(info["target"])[1].lower()

                prox_range = range(max(0, pi - proximity_window), min(len(para_nodes), pi + proximity_window + 1))
                near_question = any(idx in question_paras for idx in prox_range)
                preferred_rasters = (".png", ".jpg", ".jpeg", ".bmp", ".gif", ".webp")
                vector_formats = (".wmf", ".emf", ".svg")

                selected = False
                reasons = []

                if any(k in alt_text_l for k in KEYWORD_IN_ALT):
                    selected = True
                    reasons.append("has_alt_keyword")

                if ext in vector_formats:
                    selected = True
                    reasons.append("vector_auto_select")

                if not selected and near_question:
                    if ext in preferred_rasters:
                        selected = True
                        reasons.append("near_question_and_raster")
                    elif size >= large_size_bytes:
                        selected = True
                        reasons.append(f"near_question_and_large_size_{size}")
                    else:
                        reasons.append("near_question_but_no_strong_signal")

                if not selected and size >= (large_size_bytes * VERY_LARGE_FACTOR):
                    selected = True
                    reasons.append("auto_select_very_large_global")

                saved_path_out = None
                if sha1 in canonical_by_sha1:
                    saved_path_out = canonical_by_sha1[sha1]
                    selected = True
                    reasons.append("duplicate_of_canonical")
                    extracted_map[img_rel] = {"path": saved_path_out, "sha1": sha1}

                if selected:
                    if img_rel in extracted_map:
                        saved_path_out = extracted_map[img_rel]["path"]
                        reasons.append("reused_existing")
                    elif saved_path_out:
                        extracted_map[img_rel] = {"path": saved_path_out, "sha1": sha1}
                    else:
                        # write raw to media_out
                        orig_name = os.path.basename(img_rel)
                        # sanitize name (remove path components)
                        base_name = os.path.splitext(orig_name)[0]
                        extn = os.path.splitext(orig_name)[1]
                        candidate = f"{base_name}{extn}"
                        out_path = os.path.join(media_out, candidate)
                        if os.path.exists(out_path):
                            i = 1
                            while os.path.exists(os.path.join(media_out, f"{base_name}_{i}{extn}")):
                                i += 1
                            out_path = os.path.join(media_out, f"{base_name}_{i}{extn}")
                        try:
                            with open(out_path, "wb") as wf:
                                wf.write(raw)
                        except Exception as e:
                            reasons.append("write_failed")
                            if debug:
                                print("[error] Không thể ghi file:", out_path, e)
                            saved_path_out = None
                        else:
                            saved_path_out = out_path
                            reasons.append("extracted_now")
                            if debug:
                                print(f"[info] Extracted {img_rel} -> {saved_path_out}")

                            # convert to PNG if requested and format is convertible
                            # expand list to include .wmf, .emf, .svg, .bmp, .tif, .tiff, etc.
                            convertible_exts = (".bmp", ".tif", ".tiff", ".wmf", ".emf", ".svg", ".pcx", ".ico")
                            if convert_to_png_flag and ext in convertible_exts:
                                png_path = convert_to_png(saved_path_out, media_out, imagemagick_cmd=imagemagick_cmd, debug=debug)
                                if png_path:
                                    # remove original if convert succeeded and ext is not png
                                    try:
                                        if saved_path_out != png_path and os.path.exists(saved_path_out):
                                            os.remove(saved_path_out)
                                    except Exception:
                                        pass
                                    saved_path_out = png_path
                                    reasons.append("converted_to_png")
                                    if debug:
                                        print(f"[info] Converted {img_rel} -> {png_path}")
                                else:
                                    reasons.append("conversion_failed")
                                    if debug:
                                        print(f"[warn] Conversion failed for {out_path}")

                            # register canonical by sha1
                            if sha1 not in canonical_by_sha1 and saved_path_out:
                                canonical_by_sha1[sha1] = saved_path_out

                            extracted_map[img_rel] = {"path": saved_path_out, "sha1": sha1}

                # if canonical exists globally, override saved_path_out
                if sha1 in canonical_by_sha1:
                    saved_path_out = canonical_by_sha1[sha1]

                images_info.append({
                    "rId": rid,
                    "img_rel": img_rel,
                    "ext": ext,
                    "size": size,
                    "sha1": sha1,
                    "alt": alt_text,
                    "selected": bool(selected),
                    "saved_path": saved_path_out,
                    "reasons": reasons
                })

            # find images in runs (drawingml blip)
            for blip in p.findall(".//a:blip", NAMESPACES):
                rid = blip.get(_r_attr("embed"))
                _process_rid(rid, blip)

            # legacy VML
            for vml in p.findall(".//v:imagedata", NAMESPACES):
                rid = vml.get(_r_attr("id")) or vml.get("id")
                _process_rid(rid, vml)

            if images_info:
                images_info = _merge_images_by_hash(images_info, canonical_by_sha1, debug=debug)

            if text or math_list or images_info:
                results.append({
                    "para_index": pi,
                    "text": text,
                    "math": math_list,
                    "images": images_info
                })

    base_name = os.path.splitext(os.path.basename(docx_path))[0]
    out_json = os.path.join(out_folder, base_name + ".selective.json")
    os.makedirs(out_folder, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as jf:
        json.dump(results, jf, ensure_ascii=False, indent=2)

    if debug:
        print("[done] Đã ghi JSON ra", out_json)
    return results

if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python extract_docx_selective_strict.py path/to/file.docx")
        sys.exit(1)

    # Example invocation. Replace with sys.argv[1] or pass via CLI.
    extract_docx_selective_strict(
        # sys.argv[1],
        # r"D:\_code\DATN\data\test_data\07_ThptCbq_ThptTdn-PhongBaoBui.docx",
        r"D:\_code\DATN\data\test_data\01_TdtntTayNguyen_Ntt.docx",
        out_folder="extracted",
        convert_to_png_flag=True,
        proximity_window=3,
        min_size_bytes=5_000,
        large_size_bytes=20_000,
        debug=True
    )


<>:44: SyntaxWarning: invalid escape sequence '\W'
<>:44: SyntaxWarning: invalid escape sequence '\W'
C:\Users\nguye\AppData\Local\Temp\ipykernel_144\1949223222.py:44: SyntaxWarning: invalid escape sequence '\W'
  """


[info] rels_map size: 452
[info] paragraph 12 nghi là câu hỏi: Câu 1: Trong điều kiện chuẩn về nhiệt độ và áp suất thì
[info] paragraph 17 nghi là câu hỏi: Câu 2: Khi truyền nhiệt cho một khối khí thì khối khí có thể
[info] paragraph 20 nghi là câu hỏi: Câu 3: Câu nào sau đây nói về nhiệt lượng là không đúng?
[info] paragraph 25 nghi là câu hỏi: Câu 4: Phóng xạ là
[info] paragraph 30 nghi là câu hỏi: Câu 5: Phát biểu nào sau đây là không đúng khi nói về điện từ trường
[info] paragraph 35 nghi là câu hỏi: Câu 6: Ban đầu một mẫu chất phóng xạ nguyên chất có hạt nhân. Biết chu kì bán rã
[info] paragraph 37 nghi là câu hỏi: Câu 7: Trên hình bên là hai đường đẳng nhiệt của cùng một lượng khí lý tưởng ở h
[info] paragraph 41 nghi là câu hỏi: Câu 8: Trong quá trình truyền sóng điện từ, vector cường độ điện trường  và vect
[info] paragraph 46 nghi là câu hỏi: Câu 9: Trong hai nhiệt lượng kế có chứa hai chất lỏng khác nhau ở hai nhiệt độ b
[info] paragraph 48 nghi là câu hỏi: Câu 10: Cuộn thứ c

In [ ]:
# === 1. Load mô hình PhoBERT đã fine-tune ===
model_path = "./bert_question_answer_classifier"
clf = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path,  # ✅ dùng tokenizer của PhoBERT fine-tune
    return_all_scores=False,
    device=0,  # GPU = 0, CPU = -1
    batch_size=32   # ⭐ Tối ưu GPU
)



# === 3. Hàm xử lý từng file ===
def classify_docx(docx_path):
    paragraphs = extract_paragraphs_from_docx(docx_path)
    label_mapping = {"LABEL_0": "question", "LABEL_1": "answer"}

    classified = []
    for text in paragraphs:
        try:
            pred = clf(text)[0]
            label = label_mapping.get(pred["label"], pred["label"])
            classified.append({"text": text, "label": label})
        except Exception as e:
            classified.append({"text": text, "label": "error"})
            print(f"Lỗi khi xử lý: {text[:40]} -> {e}")

    # Gom nhóm question → answer
    grouped = []
    current_question = None
    current_answers = []

    for item in classified:
        text = item["text"]
        label = item["label"]

        if label == "question":
            if current_question and current_answers:
                grouped.append({
                    "question": current_question,
                    "answer": current_answers
                })
            current_question = text
            current_answers = []

        elif label == "answer":
            if not re.match(r"^[A-D][\.\)]", text):
                prefix = chr(65 + len(current_answers)) + ". "
                text = prefix + text
            current_answers.append(text)

    if current_question and current_answers:
        grouped.append({
            "question": current_question,
            "answer": current_answers
        })

    # Trả về DataFrame
    if grouped:
        df = pd.DataFrame(grouped)
        df["answer"] = df["answer"].apply(lambda x: str(x))
        return df
    else:
        print(f"⚠️ Không tìm thấy question/answer trong file: {os.path.basename(docx_path)}")
        return pd.DataFrame(columns=["question", "answer"])

# === 4. Xử lý nhiều file DOCX và gộp thành 1 CSV ===
input_folder = "test_data/"  # 📂 thư mục chứa các file .docx của bạn
output_csv = "src_data/all_questions_combined_2.csv"

# lấy tất cả file .docx
docx_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".docx")]
print(f"📄 Tìm thấy {len(docx_files)} file DOCX.")

# xử lý từng file
all_dataframes = []
for path in docx_files:
    print(f"🔹 Đang xử lý: {os.path.basename(path)}")
    df = classify_docx(path)
    if not df.empty:
        df["source_file"] = os.path.basename(path)
        all_dataframes.append(df)

# gộp tất cả
if all_dataframes:
    final_df = pd.concat(all_dataframes, ignore_index=True)
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"✅ Hoàn tất! Tổng {len(final_df)} câu hỏi. File lưu tại: {output_csv}")
else:
    print("⚠️ Không có dữ liệu hợp lệ nào được xuất.")


Device set to use cuda:0


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


: 